# PaceAlgo ML — Notebook 02: Feature Engineering

**Zweck:** Aus den OHLCV-Parquets (aus Notebook 01) eine Feature-Matrix pro Symbol/Timeframe berechnen.

**Architektur-Notiz:**
- **Raw OHLCV:** lebt in Drive (`MyDrive/pace-algo/data_cache/raw/`)
- **Processed Features:** lebt LOKAL in Colab (`/content/processed/`) weil Drive bei sustained heavy I/O abstürzt
- **Optional:** finale Sync zu Drive am Ende (Section 6)

**Pro Bar werden ~37 Features berechnet:**
- **Trend (7):** EMA distances, slopes, alignment, ADX
- **Momentum (5):** RSI, StochRSI, ROC, MACD-hist, momentum composite
- **Volatility (5):** ATR%, ATR percentile, BB width, vol compression, realized vol
- **Structure (4):** swing distances, BOS bullish/bearish
- **Volume (2):** relative volume, volume z-score
- **Session (2):** hour sin/cos (cyclical encoding)
- **HTF Context (6):** 1H + 4H alignment, RSI, ATR percentile
- **Macro (6):** VIX, DXY, TNX levels + 5d changes

**Laufzeit:** ~5-8 Min auf lokalem Disk (deutlich schneller als Drive).


## 1. Environment Setup

In [ ]:
import sys, os
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IS_COLAB}')

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/pace-algo'
    # Raw OHLCV stays on Drive (read-only, already there)
    DATA_RAW = Path(PROJECT_ROOT) / 'data_cache' / 'raw'
    # Processed features live LOCAL — Drive can't handle sustained writes
    DATA_PROCESSED = Path('/content/processed')
    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
    os.chdir(PROJECT_ROOT)
    # Always pull latest code from GitHub
    !rm -rf /tmp/pace-algo
    !git clone -q https://github.com/ecoNC/pace-algo.git /tmp/pace-algo
    !cp -rf /tmp/pace-algo/core/* {PROJECT_ROOT}/core/
    print('Core code updated from GitHub')
else:
    PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)
    from core.config import DATA_RAW, DATA_PROCESSED

sys.path.insert(0, PROJECT_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('core'):
        del sys.modules[mod]
print(f'Project root: {PROJECT_ROOT}')
print(f'Raw OHLCV (read):     {DATA_RAW}')
print(f'Processed (write):    {DATA_PROCESSED}')

In [ ]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

from core.config import (
    CRYPTO_SYMBOLS, FX_SYMBOLS, METAL_SYMBOLS,
    PRIMARY_TIMEFRAMES, HTF_CONTEXT_TIMEFRAMES,
)
from core.features import compute_features, attach_macro, attach_htf_context

ALL_SYMBOLS = CRYPTO_SYMBOLS + FX_SYMBOLS + METAL_SYMBOLS
print(f'Will process {len(ALL_SYMBOLS)} symbols × {len(PRIMARY_TIMEFRAMES)} primary TFs')
print(f'Symbols: {ALL_SYMBOLS}')

## 2. Load Macro (shared across symbols)

In [ ]:
macro_path = DATA_RAW / 'macro_daily.parquet'
if macro_path.exists():
    macro = pd.read_parquet(macro_path)
    print(f'Macro loaded: {len(macro):,} daily obs, columns: {list(macro.columns)}')
    display(macro.tail(3))
else:
    macro = pd.DataFrame()
    print('No macro daily found — features will be computed without macro')

## 3. Compute Features — Resilient Write Pattern

**Resilience tactics:**
- Write to `/content/` (local Colab disk) — fast, no Drive timeouts
- Per-symbol HTF cached in dict so we don't recompute
- Already-cached files skipped on re-run (resume after crash)
- Each save in try/except — one file fail doesn't break the loop

In [ ]:
def load_ohlcv(symbol: str, tf: str):
    p = DATA_RAW / f'{symbol}_{tf}.parquet'
    if not p.exists():
        return None
    return pd.read_parquet(p)

summary_rows = []
for symbol in tqdm(ALL_SYMBOLS, desc='Symbols'):
    # ── Step 1: HTF context features (1H + 4H) — computed once per symbol ──
    htf_features = {}
    for htf_tf in HTF_CONTEXT_TIMEFRAMES:
        df_htf = load_ohlcv(symbol, htf_tf)
        if df_htf is None or df_htf.empty:
            print(f'  WARN {symbol} {htf_tf}: missing OHLCV — skip HTF for this symbol')
            htf_features[htf_tf] = pd.DataFrame()
        else:
            htf_features[htf_tf] = compute_features(df_htf)

    # ── Step 2: Primary TFs (5M + 15M) ──
    for tf in PRIMARY_TIMEFRAMES:
        out_path = DATA_PROCESSED / f'{symbol}_{tf}_features.parquet'
        if out_path.exists():
            print(f'  skip {symbol} {tf} (cached)')
            continue

        df = load_ohlcv(symbol, tf)
        if df is None or df.empty:
            print(f'  SKIP {symbol} {tf}: no OHLCV')
            continue

        feats = compute_features(df)
        feats = attach_htf_context(
            feats,
            htf_features.get('1h', pd.DataFrame()),
            htf_features.get('4h', pd.DataFrame()),
        )
        feats = attach_macro(feats, macro)
        feats['symbol'] = symbol
        feats['timeframe'] = tf

        # Save with retry — if first attempt fails, try once more
        saved = False
        for attempt in range(2):
            try:
                feats.to_parquet(out_path, compression='zstd')
                saved = True
                break
            except Exception as exc:
                print(f'    write attempt {attempt+1} failed: {type(exc).__name__}: {str(exc)[:100]}')

        if not saved:
            print(f'  FAIL {symbol} {tf}: could not save after retries')
            continue

        feat_cols = [c for c in feats.columns if c not in ('symbol', 'timeframe')]
        valid_rows = feats.dropna(subset=feat_cols).shape[0]
        summary_rows.append({
            'symbol': symbol, 'tf': tf,
            'total_rows': len(feats),
            'valid_rows': valid_rows,
            'num_features': len(feat_cols),
            'size_MB': round(out_path.stat().st_size / 1024**2, 2),
        })
        print(f'  OK  {symbol} {tf}: {valid_rows:,}/{len(feats):,} valid × {len(feat_cols)} features  ({summary_rows[-1]["size_MB"]} MB)')

    # Free HTF dict for memory
    htf_features.clear()

print('\nDone.')

## 4. Summary Table

In [ ]:
# Rebuild summary from disk (works even if loop didn't finish in one pass)
processed_files = sorted(DATA_PROCESSED.glob('*_features.parquet'))
summary_disk = []
for p in processed_files:
    df = pd.read_parquet(p, columns=['symbol', 'timeframe'])
    feat_count = len(pd.read_parquet(p, columns=None).columns) - 2  # minus symbol, timeframe
    summary_disk.append({
        'file': p.name,
        'rows': len(df),
        'num_features': feat_count,
        'size_MB': round(p.stat().st_size / 1024**2, 2),
    })
sdf = pd.DataFrame(summary_disk)
display(sdf)
print(f'\nProcessed files on disk: {len(sdf)}')
print(f'Total size: {sdf["size_MB"].sum():.1f} MB')
print(f'Total rows: {sdf["rows"].sum():,}')
expected = len(ALL_SYMBOLS) * len(PRIMARY_TIMEFRAMES)
print(f'Expected files: {expected} ({len(ALL_SYMBOLS)} symbols × {len(PRIMARY_TIMEFRAMES)} TFs)')

## 5. Sanity Check — BTC 5M

In [ ]:
sample_path = DATA_PROCESSED / 'BTCUSDT_5m_features.parquet'
if sample_path.exists():
    sample = pd.read_parquet(sample_path)
    feature_cols = [c for c in sample.columns if c not in ('symbol', 'timeframe')]
    print(f'Shape: {sample.shape[0]:,} rows × {len(feature_cols)} features')
    print(f'Date range: {sample.index[0]} → {sample.index[-1]}')
    print(f'\nFeature list ({len(feature_cols)}):')
    for i, c in enumerate(feature_cols):
        print(f'  {i+1:2d}. {c}')
    print(f'\nDescriptive stats (last 100k bars):')
    display(sample[feature_cols].tail(100_000).describe().T[['mean','std','min','max']].round(3))
    print(f'\nNaN count per feature:')
    nan_counts = sample[feature_cols].isna().sum().sort_values(ascending=False)
    display(nan_counts[nan_counts > 0])
else:
    print(f'BTC 5M features not found at {sample_path}')

## 6. Sync to Drive (auto-backup)

Processed Features werden zu Drive kopiert damit Notebook 03+ sie auch nach Colab Runtime-Restart noch findet. Sync via `rsync` — schnell, resumable.

Falls Drive während Sync abstürzt: Cell nochmal ausführen.


In [ ]:
SYNC_TO_DRIVE = True   # auto-backup so NB 03+ can resume after runtime restart

if SYNC_TO_DRIVE and IS_COLAB:
    drive_target = Path(PROJECT_ROOT) / 'data_cache' / 'processed'
    drive_target.mkdir(parents=True, exist_ok=True)
    print(f'Syncing {DATA_PROCESSED} -> {drive_target}')
    print('This makes notebook 03+ work even after Colab runtime restart.')
    !rsync -ah {DATA_PROCESSED}/ {drive_target}/
    print('Sync done. Files in Drive backup:')
    !ls -lh {drive_target} | head -20
else:
    print('Sync disabled. Processed files only in /content/processed/')
    print('WARNING: If you restart Colab runtime, NB 03+ will fail to find these files.')


---

## Nächster Schritt

→ `03_asset_clustering.ipynb`

K-Means Clustering der Symbole auf Volatilitäts-/Trend-Eigenschaften → datengetriebene Asset-Klassen.

Notebook 03 liest aus `/content/processed/` direkt — also direkt nach diesem Notebook starten ohne Runtime-Restart.